# FAC_NB_Carga_Generica — carga genérica, completa o incremental, dirigida por parámetros

Un **único cuaderno** carga cualquier tabla descrita en el catálogo `fac_catalogo_ingesta`. El pipeline lo invoca una vez por fila del catálogo (ForEach), pasando los valores como parámetros. Todo es sintético y didáctico, no una implementación de producción.

## Flujo

```text
parámetros → validar → leer origen → [incremental: filtrar columna_marca > ultima_marca]
          → escribir Delta (completa = overwrite | incremental = MERGE insert-only por llave)
          → calcular nueva marca → bitácora → actualizar catálogo (MERGE, solo si todo salió bien)
          → notebookutils.notebook.exit(JSON)
```

## Parámetros

| Parámetro | Significado |
|---|---|
| `tabla` | Tabla Delta destino, **sin** esquema (llave en catálogo y bitácora) |
| `ruta_origen` | Archivo o carpeta bajo `Files` del Lakehouse predeterminado |
| `formato` | `csv`, `parquet`, `json` o `delta` |
| `tipo_carga` | `completa` o `incremental` |
| `columna_marca` | Columna fecha/hora para la marca de agua (incremental) |
| `columna_clave` | Llave de negocio para que el incremental sea idempotente |
| `ultima_marca` | Última marca cargada (`yyyy-MM-ddTHH:mm:ss`); vacía = carga inicial |
| `destino` | Esquema destino: `auto`, `ninguno` o un nombre |
| `run_id` | Identificador de la corrida (el pipeline pasa `@pipeline().RunId`); vacío = se genera uno |
| `forzar_completa` | `True` ignora `tipo_carga` y recarga todo (y reinicia la marca) |

## Decisiones de diseño

- **Completa = `overwrite`.** Las tablas maestras son pequeñas y se quiere una foto fiel del origen; reejecutar deja el mismo resultado. Protección: si el origen viene **vacío** no se sobrescribe el destino (podría ser una exportación fallida) y se registra una advertencia.
- **Incremental = `MERGE` insert-only por `columna_clave`**, no `append`. Escribir los datos y avanzar la marca **no es una operación atómica**: si falla entre ambas, la siguiente corrida relee las mismas filas. Con `append` se duplicarían; con el MERGE las filas que ya existen se ignoran. Así la carga es *al menos una vez* con escritura idempotente, o sea efectivamente una vez. Es insert-only porque los movimientos bancarios ya aplicados no cambian; si las filas pudieran cambiar habría que añadir `WHEN MATCHED THEN UPDATE` y deduplicar el origen.
- **La marca solo avanza si la escritura fue exitosa**, y se calcula sobre valores de fecha **válidos**: el dataset trae fechas imposibles a propósito (`2026-09-31T25:61:00`); como texto serían la "máxima" y corromperían la marca.
- **Filtro estricto (`>`)**. Es la regla clásica de la marca de agua. Limitación conocida: una fila tardía con la misma marca exacta no se cargaría; en producción se suele reprocesar una ventana de seguridad apoyándose en el MERGE idempotente.
- **Columnas de auditoría** `_fac_run_id` y `_fac_cargado_en` en cada tabla destino: permiten ver qué corrida cargó cada fila.
- Cuando algo falla, **se deja constancia en la bitácora y luego el cuaderno falla** (para que el pipeline lo muestre en rojo).

In [ ]:
# Celda de parámetros: el pipeline sobrescribe estos valores (Base parameters de la actividad Notebook).
tabla = ""
ruta_origen = ""
formato = "csv"
tipo_carga = "completa"
columna_marca = ""
columna_clave = ""
ultima_marca = ""
destino = "auto"
run_id = ""
forzar_completa = False

## Paso 1 — Preparación: constantes, utilidades y normalización de parámetros

Se aplica el mismo patrón de nombres que el resto de los cuadernos (`fac_*`, esquema `fac` si el Lakehouse tiene esquemas). Los parámetros llegan como texto desde el pipeline, así que se normalizan: se recortan espacios, `forzar_completa` se interpreta como booleano y, si no hay `run_id`, se genera uno.

In [ ]:
import json
import re
import uuid
import datetime
from pyspark.sql import functions as F

PATRON_IDENT = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,63}\Z")


def validar_ident(valor, campo):
    """Los nombres se interpolan en SQL: solo se aceptan identificadores simples."""
    if not isinstance(valor, str) or PATRON_IDENT.fullmatch(valor) is None:
        raise ValueError(
            f"{campo}: se esperaba un identificador (letra inicial; letras, números y _); se recibió {valor!r}"
        )
    return valor


def resolver_esquema(destino):
    """auto = 'fac' si el Lakehouse tiene esquemas habilitados, si no ninguno; 'ninguno' o un nombre explícito."""
    d = (destino or "auto").strip()
    if d.lower() == "ninguno":
        return ""
    if d.lower() == "auto":
        try:
            nombres = {str(r[0]).lower() for r in spark.sql("SHOW SCHEMAS").collect()}
        except Exception:
            nombres = set()
        return "fac" if "dbo" in nombres else ""
    return validar_ident(d, "destino")


def calificar(esquema, tabla):
    return f"{esquema}.{tabla}" if esquema else tabla

import notebookutils

FORMATOS = ("csv", "parquet", "json", "delta")
TIPOS = ("completa", "incremental")
PATRON_MARCA = re.compile(r"\d{4}-\d{2}-\d{2}([T ]\d{2}:\d{2}:\d{2}(\.\d+)?)?(Z|[+-]00:?00)?\Z")
FORMATO_MARCA = "yyyy-MM-dd'T'HH:mm:ss"


def como_bool(valor):
    return str(valor).strip().lower() in ("true", "1", "si", "sí", "yes")


def ahora_utc():
    return datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)


def normalizar_marca(texto):
    """Misma forma ISO siempre (yyyy-MM-ddTHH:mm:ss) para poder comparar como texto.
    El pipeline puede devolver la marca con 'Z' o con fracciones de segundo; se toleran porque la marca siempre es hora UTC de la sesión."""
    texto = (texto or "").strip().replace(" ", "T")
    if not texto:
        return ""
    if PATRON_MARCA.fullmatch(texto) is None:
        raise ValueError(f"ultima_marca con formato inválido: {texto!r} (se espera yyyy-MM-ddTHH:mm:ss)")
    texto = re.sub(r"(\.\d+)?(Z|[+-]00:?00)?\Z", "", texto)
    return texto if len(texto) > 10 else texto + "T00:00:00"


tabla = (tabla or "").strip()
ruta_origen = (ruta_origen or "").strip()
formato = (formato or "").strip().lower()
columna_marca = (columna_marca or "").strip()
columna_clave = (columna_clave or "").strip()
FORZAR = como_bool(forzar_completa)
TIPO_EFECTIVO = "completa" if FORZAR else (tipo_carga or "").strip().lower()
RUN_ID = (run_id or "").strip() or str(uuid.uuid4())
print(f"run_id={RUN_ID} | tabla={tabla} | tipo efectivo={TIPO_EFECTIVO} | forzar_completa={FORZAR}")

## Paso 2 — Validaciones (antes de leer o escribir nada)

Se comprueba: identificadores y valores permitidos, que exista un Lakehouse predeterminado, que **existan las tablas de control** (catálogo y bitácora; las crea `FAC_NB_Preparar_Control`), que la tabla **figure en el catálogo** (si no, se avisa en el mensaje) y que el formato de `ultima_marca` sea correcto. Si el origen no existe, falla con un mensaje claro en el paso de lectura.

In [ ]:
def validar_parametros():
    validar_ident(tabla, "tabla")
    if formato not in FORMATOS:
        raise ValueError(f"formato debe ser uno de {FORMATOS}; se recibió {formato!r}")
    if TIPO_EFECTIVO not in TIPOS:
        raise ValueError(f"tipo_carga debe ser uno de {TIPOS}; se recibió {tipo_carga!r}")
    if not (ruta_origen.startswith("Files/") or ruta_origen.startswith("abfss://")):
        raise ValueError("ruta_origen debe empezar por 'Files/' (Lakehouse predeterminado) o 'abfss://'")
    if columna_marca:
        validar_ident(columna_marca, "columna_marca")
    if columna_clave:
        validar_ident(columna_clave, "columna_clave")
    if TIPO_EFECTIVO == "incremental" and not (columna_marca and columna_clave):
        raise ValueError("una carga incremental necesita columna_marca y columna_clave")
    if not notebookutils.runtime.context.get("defaultLakehouseId"):
        raise RuntimeError("Adjunta el Lakehouse del laboratorio como predeterminado y reinicia la sesión.")
    for t in (CAT, BIT):
        if not spark.catalog.tableExists(t):
            raise RuntimeError(f"No existe la tabla de control {t}. Ejecuta primero FAC_NB_Preparar_Control.")


ESQUEMA = resolver_esquema(destino)
CAT = calificar(ESQUEMA, "fac_catalogo_ingesta")
BIT = calificar(ESQUEMA, "fac_bitacora_ingesta")
print("Esquema:", ESQUEMA or "(ninguno)")

## Paso 3 — Leer el origen y preparar la marca de agua

- Los CSV se leen **todo como texto** (como en bronce de la Sesión 2) y en modo estricto (`FAILFAST`): una fila mal formada detiene la carga en vez de convertirse en nulos en silencio. Los campos vacíos (por ejemplo, el `canal` vacío intencional) se conservan como **texto vacío**: Spark los lee como NULL (un CSV no tiene NULL), así que tras leer se rellenan con `''`. También se limpia un posible BOM en los nombres de columna.
- La marca se convierte con `try_to_timestamp`: un valor imposible da `NULL` en lugar de romper o, peor, ganar la comparación como texto.
- Con `ultima_marca` vacía es una **carga inicial**: se leen todas las filas (incluidas las de fecha inválida, que bronce conserva para que la capa plata las ponga en cuarentena). Con marca, solo pasan las filas con `columna_marca > ultima_marca`; las de fecha inválida no pasan y se cuentan en el mensaje.

In [ ]:
def leer_origen():
    ruta = ruta_origen.rstrip("/")
    if not notebookutils.fs.exists(ruta):
        raise FileNotFoundError(f"El origen no existe: {ruta}. ¿Se cargaron los CSV de la Sesión 1 y se ejecutó FAC_NB_Preparar_Control?")
    lector = spark.read.format(formato)
    if formato == "csv":
        lector = lector.option("header", "true").option("inferSchema", "false").option("mode", "FAILFAST")
    df = lector.load(ruta_origen)
    df = df.toDF(*[c.replace("\ufeff", "").strip() for c in df.columns])
    if formato == "csv":
        # Un CSV no tiene NULL: Spark lee los campos vacíos como NULL y aquí se devuelven a texto vacío (como en bronce de la Sesión 2).
        df = df.na.fill("")
    return df


def expresion_marca(df, columna):
    if columna not in df.columns:
        raise ValueError(f"columna '{columna}' no existe en el origen; columnas: {df.columns}")
    tipo = dict(df.dtypes)[columna]
    if tipo.startswith("timestamp") or tipo == "date":
        return F.col(columna).cast("timestamp")
    return F.expr(f"try_to_timestamp(`{columna}`)")


def perfilar(df, col_marca, col_clave):
    """Una sola pasada: filas, fechas válidas, marca máxima (como texto ISO) y llaves nulas."""
    aggs = [F.count(F.lit(1)).alias("n")]
    if col_marca:
        m = expresion_marca(df, col_marca)
        aggs += [F.count(m).alias("n_marca_valida"), F.date_format(F.max(m), FORMATO_MARCA).alias("max_marca")]
    if col_clave:
        if col_clave not in df.columns:
            raise ValueError(f"columna_clave '{col_clave}' no existe en el origen; columnas: {df.columns}")
        aggs += [F.sum(F.when(F.col(col_clave).isNull() | (F.trim(F.col(col_clave).cast("string")) == ""), 1).otherwise(0)).alias("n_clave_nula")]
    return df.agg(*aggs).first().asDict()

## Paso 4 — Escribir en Delta

- **Completa:** `overwrite` con `overwriteSchema`, y se comprueba que el conteo de la tabla coincida con lo leído.
- **Incremental:** si la tabla no existe se crea con la primera carga; si existe, `MERGE ... WHEN NOT MATCHED THEN INSERT *` por la llave. Las filas ya presentes se ignoran (idempotencia). Las filas escritas se miden como la diferencia de conteos antes y después.

In [ ]:
def agregar_auditoria(df, run):
    return df.withColumn("_fac_run_id", F.lit(run)).withColumn("_fac_cargado_en", F.current_timestamp())


def escribir_completa(df, tabla_q, run, esperadas):
    agregar_auditoria(df, run).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla_q)
    final = spark.table(tabla_q).count()
    if final != esperadas:
        raise RuntimeError(f"{tabla_q}: se esperaban {esperadas} filas tras la carga completa y hay {final}")
    return final, "overwrite"


def escribir_incremental(df, tabla_q, clave, run):
    salida = agregar_auditoria(df, run)
    if not spark.catalog.tableExists(tabla_q):
        salida.write.format("delta").saveAsTable(tabla_q)
        return spark.table(tabla_q).count(), "tabla creada con la carga inicial"
    antes = spark.table(tabla_q).count()
    vista = f"_fac_src_{tabla}"
    salida.createOrReplaceTempView(vista)
    spark.sql(f"""
        MERGE INTO {tabla_q} AS t
        USING {vista} AS s
        ON t.`{clave}` = s.`{clave}`
        WHEN NOT MATCHED THEN INSERT *
    """)
    despues = spark.table(tabla_q).count()
    return despues - antes, "merge insert-only"

## Paso 5 — Bitácora y catálogo

- `registrar_bitacora` agrega **una fila por corrida y tabla**, también cuando hubo error.
- `actualizar_catalogo` hace un `MERGE` que solo toca `ultima_marca` de la fila de esa tabla y luego lee el valor para comprobar que quedó guardado. Si la tabla no está en el catálogo (por ejemplo, una ejecución manual con valores sueltos) no hay fila que actualizar y se deja constancia en el mensaje.

In [ ]:
SCHEMA_BIT = ("run_id STRING, tabla STRING, inicio TIMESTAMP, fin TIMESTAMP, filas_leidas BIGINT, "
              "filas_escritas BIGINT, marca_antes STRING, marca_despues STRING, estado STRING, mensaje STRING")


def registrar_bitacora(r):
    fila = (r["run_id"], r["tabla"], r["inicio"], r["fin"], int(r["filas_leidas"]), int(r["filas_escritas"]),
            r["marca_antes"], r["marca_despues"], r["estado"], (r["mensaje"] or "")[:1000])
    spark.createDataFrame([fila], SCHEMA_BIT).write.format("delta").mode("append").saveAsTable(BIT)


def actualizar_catalogo(nueva_marca):
    spark.createDataFrame([(tabla, nueva_marca)], "tabla_destino STRING, ultima_marca STRING").createOrReplaceTempView("_fac_marca")
    spark.sql(f"""
        MERGE INTO {CAT} AS c
        USING _fac_marca AS s
        ON c.tabla_destino = s.tabla_destino
        WHEN MATCHED THEN UPDATE SET c.ultima_marca = s.ultima_marca
    """)
    guardada = spark.table(CAT).where(F.col("tabla_destino") == tabla).select("ultima_marca").first()
    if guardada is None:
        return "la tabla no está en el catálogo: la marca no se persistió"
    if guardada[0] != nueva_marca:
        raise RuntimeError(f"la marca guardada en el catálogo ({guardada[0]!r}) no coincide con la calculada ({nueva_marca!r})")
    return ""

## Paso 6 — Orquestación de la carga

`ejecutar()` encadena todo y **siempre devuelve un diccionario** con el resultado (no lanza excepciones hacia afuera), para que el paso siguiente pueda registrar la bitácora incluso si algo falló.

Estados: `OK`, `SIN_DATOS` (incremental sin filas nuevas: no se escribe nada ni cambia la marca), `ADVERTENCIA` (completa con origen vacío: no se sobrescribe) y `ERROR`.

**Cero filas no rompe nada.** Se cuentan las filas leídas y las escritas, y para una carga completa y para un MERGE se comprueba que el conteo del destino sea coherente.

In [ ]:
def ejecutar():
    r = dict(run_id=RUN_ID, tabla=tabla, inicio=ahora_utc(), fin=None, tipo_carga=TIPO_EFECTIVO,
             filas_leidas=0, filas_escritas=0, marca_antes=None, marca_despues=None, estado="ERROR", mensaje="")
    notas = []
    df_carga = None
    try:
        validar_parametros()
        if spark.table(CAT).where(F.col("tabla_destino") == tabla).count() == 0:
            notas.append("la tabla no figura en el catálogo (ejecución manual): la marca no se persistirá")
        usa_marca = TIPO_EFECTIVO == "incremental"
        marca_antes = "" if FORZAR else normalizar_marca(ultima_marca)
        r["marca_antes"] = marca_antes or None
        r["marca_despues"] = marca_antes or None
        tabla_q = calificar(ESQUEMA, tabla)

        df = leer_origen()
        origen = perfilar(df, columna_marca, None)
        notas.append(f"origen={origen['n']} filas")
        if columna_marca:
            invalidas = origen["n"] - origen["n_marca_valida"]
            if invalidas:
                notas.append(f"{invalidas} filas con fecha inválida en {columna_marca} (excluidas del cálculo de la marca)")

        if usa_marca and marca_antes:
            if spark.sql(f"SELECT try_to_timestamp('{marca_antes}')").first()[0] is None:
                raise ValueError(f"ultima_marca no es una fecha válida: {marca_antes!r}")
            df_carga = df.where(expresion_marca(df, columna_marca) > F.expr(f"try_to_timestamp('{marca_antes}')"))
            notas.append(f"filtro: {columna_marca} > {marca_antes}")
        else:
            df_carga = df
            notas.append("carga inicial o completa: sin filtro de marca")
        df_carga = df_carga.cache()

        perfil = perfilar(df_carga, columna_marca, columna_clave if usa_marca else None)
        n = int(perfil["n"])
        r["filas_leidas"] = n
        if usa_marca and int(perfil.get("n_clave_nula") or 0) > 0:
            raise ValueError(f"hay {int(perfil['n_clave_nula'])} filas con {columna_clave} nulo o vacío: el MERGE no podría garantizar idempotencia")

        if n == 0:
            if usa_marca:
                r["estado"] = "SIN_DATOS"
                notas.append("no hay filas nuevas: no se escribe nada y la marca no cambia")
            else:
                r["estado"] = "ADVERTENCIA"
                notas.append("el origen está vacío: NO se sobrescribe el destino")
        else:
            if usa_marca:
                escritas, como = escribir_incremental(df_carga, tabla_q, columna_clave, RUN_ID)
                if escritas < n:
                    notas.append(f"{n - escritas} filas ya existían (se ignoran por idempotencia)")
            else:
                escritas, como = escribir_completa(df_carga, tabla_q, RUN_ID, n)
            r["filas_escritas"] = int(escritas)
            notas.append(como)
            r["estado"] = "OK"

            nueva = perfil.get("max_marca") if columna_marca else None
            if nueva:
                if not FORZAR and marca_antes and nueva < marca_antes:
                    nueva = marca_antes
                r["marca_despues"] = nueva
                if FORZAR or nueva != marca_antes:
                    try:
                        extra = actualizar_catalogo(nueva)
                    except Exception as e_cat:
                        raise RuntimeError(f"datos escritos pero la marca NO se actualizó ({e_cat}); reintentar es seguro por el MERGE idempotente")
                    if extra:
                        notas.append(extra)
    except Exception as e:
        r["estado"] = "ERROR"
        notas.append(f"{type(e).__name__}: {str(e)[:500]}")
    finally:
        if df_carga is not None:
            df_carga.unpersist()
        r["fin"] = ahora_utc()
        r["mensaje"] = " | ".join(notas)
    return r

## Paso 7 — Ejecutar, registrar y mostrar el resultado

Se ejecuta la carga y se escribe la bitácora (siempre). Si falla el registro en la bitácora, el resultado pasa a `ERROR` para que no se pierda en silencio.

In [ ]:
resultado = ejecutar()
try:
    registrar_bitacora(resultado)
except Exception as e_bit:
    resultado["estado"] = "ERROR"
    resultado["mensaje"] = (resultado["mensaje"] + f" | bitácora no escrita: {str(e_bit)[:300]}").strip(" |")

salida = {k: (v.isoformat(timespec="seconds") if isinstance(v, datetime.datetime) else v) for k, v in resultado.items()}
salida["duracion_s"] = round((resultado["fin"] - resultado["inicio"]).total_seconds(), 1)
print(json.dumps(salida, ensure_ascii=False, indent=2))

## Dónde entra Key Vault (ilustrativo, no se ejecuta)

La celda siguiente es **solo comentarios**: no se ejecuta ni necesita ningún servicio. Esta demo lee archivos del propio Lakehouse, por lo que no hay credenciales. Si el origen fuera una base relacional, así encajaría un secreto de Azure Key Vault.

In [ ]:
# =====================================================================================
# Dónde entra Key Vault (ilustrativo, no se ejecuta)
# =====================================================================================
# Regla: ninguna contraseña va en el cuaderno, en el catálogo ni como parámetro del pipeline.
# El catálogo guardaría, como mucho, el NOMBRE del secreto.
#
# 1) Leer un secreto. Firma verificada en Microsoft Learn (NotebookUtils credentials):
#        getSecret(akvName: String, secret: String): String
#    - akvName es la URL completa del vault: "https://<nombre>.vault.azure.net/"
#    - se autentica con la identidad que ejecuta el cuaderno; necesita permiso de lectura (Get)
#      sobre el secreto
#    - la salida del cuaderno intenta ocultar el valor, pero Learn aclara que no es un límite de
#      seguridad: no imprimir ni guardar el secreto
#
# vault = "https://<nombre-del-vault>.vault.azure.net/"
# usuario = notebookutils.credentials.getSecret(vault, "origen-usuario")
# clave = notebookutils.credentials.getSecret(vault, "origen-clave")
#
# 2) Lectura JDBC con el predicado incremental empujado al ORIGEN: la base filtra y Spark solo
#    recibe las filas nuevas (ultima_marca ya se valida con una expresión regular en normalizar_marca).
#
# consulta = (f"SELECT * FROM dbo.transacciones WHERE fecha_hora > '{ultima_marca}'")
# df = (spark.read.format("jdbc")
#       .option("url", "jdbc:sqlserver://<servidor>:1433;databaseName=<base>;encrypt=true")
#       .option("query", consulta)
#       .option("user", usuario)
#       .option("password", clave)
#       .load())
# # A partir de aquí el resto del cuaderno (auditoría, MERGE idempotente, marca, bitácora) no cambia.
#
# 3) Orígenes locales (on-premises) y gateways: qué dice Learn, sin asumir nada más
#    - Data Factory en Fabric (pipelines, p. ej. la actividad Copy) y Copy job pueden leer orígenes
#      locales detrás de un firewall o dentro de una red virtual mediante una gateway:
#        * gateway de datos local (on-premises data gateway): software que se instala en la red
#          local y solo requiere conexiones salientes;
#        * gateway de datos de red virtual (VNet data gateway): servicio administrado, sin instalar
#          nada; Learn la lista para Dataflow Gen2, pipelines, Copy job y mirroring.
#    - Un cuaderno de Spark abre su propia conexión de red desde el cómputo de Fabric. Las páginas
#      de Learn sobre esas gateways no lo incluyen entre los elementos que se ejecutan a través de
#      ellas. Por eso, desde un cuaderno el origen tiene que ser alcanzable por red desde Fabric
#      (por ejemplo, un endpoint permitido por el firewall del origen, o un endpoint privado
#      administrado hacia servicios de Azure compatibles).
#    - Qué opción aplica en una organización concreta depende de su red y de sus políticas:
#      confirmarlo en su entorno antes de diseñar nada.
#    - Las conexiones de Fabric pueden tomar sus credenciales de Azure Key Vault mediante
#      "Azure Key Vault references" (configuración en Manage connections and gateways).
#
# Referencias:
#   https://learn.microsoft.com/en-us/fabric/data-engineering/notebookutils/notebookutils-credentials
#   https://learn.microsoft.com/en-us/fabric/data-factory/how-to-access-on-premises-data
#   https://learn.microsoft.com/en-us/data-integration/vnet/overview
#   https://learn.microsoft.com/en-us/fabric/data-factory/copy-job-connectors
#   https://learn.microsoft.com/en-us/fabric/security/security-managed-private-endpoints-overview
#   https://learn.microsoft.com/en-us/fabric/data-factory/azure-key-vault-reference-configure

## Paso 8 — Salir con el resultado

- Si el estado es `ERROR`, el cuaderno **falla a propósito** (después de haber escrito la bitácora) para que la actividad del pipeline aparezca en rojo.
- Si no, `notebookutils.notebook.exit` devuelve el resultado como **JSON en texto**; en el pipeline se lee con `@json(activity('Cargar_Tabla').output.result.exitValue)`.

`exit()` va en su propia celda y **fuera** de cualquier `try/except`: dentro de uno, la salida no surte efecto (así lo indica Microsoft Learn). Al ejecutarlo de forma interactiva, `exit()` interrumpe el cuaderno por diseño.

In [ ]:
if resultado["estado"] == "ERROR":
    raise RuntimeError(f"Carga fallida para {tabla}: {resultado['mensaje']}")
notebookutils.notebook.exit(json.dumps(salida, ensure_ascii=False))